<a href="https://colab.research.google.com/github/PE-R2000/RFD3_Boltz_pipeline/blob/main/RFD3_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RFDiffusion3 Interactive Design Pipeline 🔨🧬

**Design, sequence, predict, and rank protein candidates in one Colab notebook.**

This notebook is built for Google Colab and keeps the full RFD3 workflow while making each step explicit and interactive.
You will load a structure, configure RFDiffusion3, generate sequences with ProteinMPNN or LigandMPNN, run Boltz-2 as an external prediction engine, and rank the final structures with a compact scoring report.

🟢 **Recommended runtime:** GPU  |  🔵 **Target platform:** Google Colab  |  🟡 **Boltz-2 installed separately from GitHub**

Run the notebook from top to bottom. The final cell contains a compact operating guide and troubleshooting notes.

In [ ]:
#@title 🔧 Install rc-foundry and model checkpoints (~ 10 min)
#@markdown This cell installs rc-foundry, prepares the Colab workspace, and downloads the RFD3, RF3, and MPNN checkpoints. Re-running the cell is safe.

import os
import subprocess
import sys
from pathlib import Path

COLAB_ROOT = Path("/content")
WORK_ROOT = COLAB_ROOT
STRUCTURE_ROOT = COLAB_ROOT / "structures"
OUTPUT_ROOT = COLAB_ROOT / "outputs"
DOWNLOAD_ROOT = COLAB_ROOT / "downloads"
CACHE_ROOT = COLAB_ROOT / "cache"
CCD_DIR = CACHE_ROOT / "ccd"
PDB_DIR = CACHE_ROOT / "pdb"
READY_FILE = COLAB_ROOT / "FOUNDRY_READY"

for directory in [WORK_ROOT, STRUCTURE_ROOT, OUTPUT_ROOT, DOWNLOAD_ROOT, CACHE_ROOT, CCD_DIR, PDB_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ["CCD_MIRROR_PATH"] = str(CCD_DIR)
os.environ["PDB_MIRROR_PATH"] = str(PDB_DIR)


def run_command(command: list[str], description: str) -> None:
    print(f"\n[{description}] {' '.join(command)}")
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is not None:
        for line in process.stdout:
            print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}: {' '.join(command)}")


if not READY_FILE.exists():
    run_command([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], "Removing torchvision to avoid Colab operator conflicts")
    run_command([sys.executable, "-m", "pip", "install", "-q", "rc-foundry[all]"], "Installing rc-foundry")
    READY_FILE.touch()
else:
    print("rc-foundry is already installed in this Colab session cache.")

run_command(["foundry", "install", "rfd3", "ligandmpnn"], "Downloading model checkpoints")

print("\nWorkspace root:", WORK_ROOT)
print("Structure directory:", STRUCTURE_ROOT)
print("Output directory:", OUTPUT_ROOT)


[Removing torchvision to avoid Colab operator conflicts] /usr/bin/python3 -m pip uninstall -y torchvision
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu

[Installing rc-foundry] /usr/bin/python3 -m pip install -q rc-foundry[all]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.0/517.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.1/31.1 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.3/402.3 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.9/205.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.5/139.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Install Boltz-2

⚠️ **Dependency conflict:** rc-foundry requires `einx>=0.1.0,<1` and Boltz requires `einx>=0.3.0`. Installing Boltz will upgrade einx to 0.3.0 (which still satisfies foundry's constraint).

**After installing Boltz-2, you MUST restart the Colab runtime** (Runtime → Restart runtime), then re-run the Install rc-foundry cell before running the Boltz prediction cells.

Boltz-2 is an external package and is not bundled with rc-foundry. The next cell installs it directly from the official repository so the ranking stage can use a separate structure prediction engine.

In [ ]:
#@title 🔮 Install Boltz-2 from GitHub (optional)
#@markdown Boltz-2 is installed from https://github.com/jwohlwend/boltz.git. After installing, you MUST restart the runtime before running Boltz cells. Re-running the cell is safe.

import sys

install_boltz_now = True #@param {type:"boolean"}

BOLTZ_READY_FILE = COLAB_ROOT / "BOLTZ_READY"
BOLTZ_GIT_URL = "git+https://github.com/jwohlwend/boltz.git"

if install_boltz_now:
    print("\n⚠️  WARNING: Installing Boltz-2 will upgrade einx from 0.1.0 to 0.3.0.")
    print("After this cell completes, you MUST restart the Colab runtime")
    print("(Runtime → Restart runtime), then re-run the Install rc-foundry cell.\n")
    if not BOLTZ_READY_FILE.exists():
        run_command([sys.executable, "-m", "pip", "install", "-q", BOLTZ_GIT_URL], "Installing Boltz-2 from GitHub")
        BOLTZ_READY_FILE.touch()
    else:
        print("Boltz-2 is already installed in this Colab session cache.")
    run_command([sys.executable, "-m", "pip", "show", "boltz"], "Verifying Boltz-2 installation")
    print("\n✅ Boltz-2 installed. Now restart the runtime before running Boltz cells.")
else:
    print("Boltz-2 installation skipped. Set install_boltz_now = True to install.")


⚠️  WARNING: Installing Boltz-2 will upgrade einx from 0.1.0 to 0.3.0.
After this cell completes, you MUST restart the Colab runtime
(Runtime → Restart runtime), then re-run the Install rc-foundry cell.


[Installing Boltz-2 from GitHub] /usr/bin/python3 -m pip install -q git+https://github.com/jwohlwend/boltz.git
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.2/399.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
#@title ── Load libraries and helper functions
#@markdown This cell defines the reusable building blocks for the notebook: structure loading, parsing form values, RFD3 execution, MPNN export, Boltz-2 YAML generation, ranking, and downloads.

import ast
import glob
import gzip
import json
import os
import re
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
import yaml
from IPython.display import HTML, clear_output, display
from atomworks.io import parse
from atomworks.io.utils.visualize import view
from biotite.database.rcsb import fetch
from biotite.sequence import ProteinSequence
from biotite.structure import get_residue_starts
from biotite.structure.info import vdw_radius_single
from biotite.structure.io import pdb, pdbx
from biotite.structure.sasa import sasa
from lightning.fabric import seed_everything
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

try:
    from rfd3.engine import DesignInputSpecification, RFD3InferenceConfig, RFD3InferenceEngine
    from rfd3.inference.symmetry.symmetry_utils import SymmetryConfig
    from rfd3.model.inference_sampler import SampleDiffusionConfig
except ImportError:
    pass

from scipy.spatial.distance import cdist

warnings.filterwarnings("ignore", module="atomworks")

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

PIPELINE_STATE = globals().get("PIPELINE_STATE", {})
AA_CODES = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLU", "GLN", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL",
}

def info_box(title: str, body: str) -> None:
    """Print a plain-text status box (no HTML)."""
    border = "─" * max(len(title), 40)
    print(f"\n┌─{border}─┐")
    print(f"│ {title}")
    print(f"├─{border}─┤")
    for line in body.split("\n"):
        print(f"│ {line}")
    print(f"└─{border}─┘\n")

def clean_text(raw):
    if raw is None: return None
    text = str(raw).strip()
    if text in {"", "None", "null", "NULL"}: return None
    return text

def parse_literal_or_text(raw):
    text = clean_text(raw)
    if text is None: return None
    lowered = text.lower()
    if lowered == "true": return True
    if lowered == "false": return False
    for parser in (json.loads, ast.literal_eval):
        try: return parser(text)
        except: continue
    return text

def parse_csv_list(raw):
    text = clean_text(raw)
    if text is None: return None
    return [item.strip() for item in text.split(",") if item.strip()]

def parse_float_vector(raw):
    text = clean_text(raw)
    if text is None: return None
    values = [float(item.strip()) for item in text.split(",") if item.strip()]
    if len(values) != 3:
        raise ValueError("Origin coordinates must contain exactly three comma-separated numbers.")
    return values

def strip_known_suffixes(path: Path) -> str:
    name = path.name
    for suffix in (".cif.gz", ".pdb.gz", ".cif", ".pdb", ".json", ".fasta"):
        if name.endswith(suffix): return name[: -len(suffix)]
    return path.stem


def load_atom_array_from_file(file_path: str | Path):
    path = Path(file_path)
    if path.name.endswith(".cif.gz") or path.suffix == ".cif":
        if path.name.endswith(".gz"):
            with gzip.open(path, "rt") as h: cif = pdbx.CIFFile.read(h)
        else: cif = pdbx.CIFFile.read(path)
        return pdbx.get_structure(cif)[0]
    if path.name.endswith(".pdb.gz") or path.suffix == ".pdb":
        if path.name.endswith(".gz"):
            with gzip.open(path, "rt") as h: p = pdb.PDBFile.read(h)
        else: p = pdb.PDBFile.read(path)
        return pdb.get_structure(p)[0]
    raise ValueError(f"Unsupported: {path}")

class StructureFetcher:
    @staticmethod
    def read_local(path): return load_atom_array_from_file(path), Path(path)
    @staticmethod
    def from_rcsb(pdb_id, file_format="cif", target_path=None):
        target_path = Path(target_path or "/content/structures/rcsb"); target_path.mkdir(parents=True, exist_ok=True)
        fetched = Path(fetch(pdb_id, format=file_format, target_path=str(target_path)))
        return load_atom_array_from_file(fetched), fetched

def build_symmetry_config(s_id, is_unsym, is_sym_motif):
    if not s_id or s_id == "None": return None
    kwargs = {"id": s_id, "is_symmetric_motif": bool(is_sym_motif)}
    if clean_text(is_unsym): kwargs["is_unsym_motif"] = clean_text(is_unsym)
    return SymmetryConfig(**kwargs)

def rename_rfd3_outputs(output_dir, design_name, batch_index):
    pattern = re.compile(r"backbone_\\d+_(\\d+)_model(...)")
    for path in output_dir.iterdir():
        match = pattern.fullmatch(path.name)
        if not match: continue
        new_name = f"{design_name}_batch_{batch_index:02d}_design_{int(match.group(1)):02d}{match.group(2)}"
        path.rename(path.with_name(new_name))

def run_rfd3_batches(design_name, rfd3_config, design_spec, num_batches, output_root):
    engine = RFD3InferenceEngine(**RFD3InferenceConfig(**rfd3_config))
    arrays, files = [], []
    for i in range(num_batches):
        b_dir = output_root / design_name / f"batch_{i:02d}" / "RFD3_designs"
        b_dir.mkdir(parents=True, exist_ok=True)
        engine.run(inputs=design_spec, n_batches=1, out_dir=str(b_dir))
        rename_rfd3_outputs(b_dir, design_name, i)
        b_files = sorted(b_dir.glob("*.cif*"))
        files.extend(b_files); arrays.append([load_atom_array_from_file(p) for p in b_files])
    return arrays, files


def get_current_structure_files():
    """Return the most recent structure files in the pipeline.

    Priority: MPNN outputs > RFD3 outputs > input structure file.
    The second return value is a string tag: 'mpnn', 'rfd3', or 'input'.
    """
    if PIPELINE_STATE.get("mpnn_cif_files"):
        return [Path(p) for p in PIPELINE_STATE["mpnn_cif_files"]], "mpnn"
    if PIPELINE_STATE.get("rfd3_files"):
        return [Path(p) for p in PIPELINE_STATE["rfd3_files"]], "rfd3"
    if PIPELINE_STATE.get("structure_file_path"):
        return [Path(PIPELINE_STATE["structure_file_path"])], "input"
    raise ValueError(
        "No structures available in the pipeline.\n"
        "Load an input structure (Load input structure cell) or run RFD3 first."
    )

def run_mpnn_pipeline(design_name, input_files, mpnn_settings, output_root):
    """Run MPNN on each structure file, passing the full mpnn_settings dict."""
    rows, mpnn_files = [], []
    root = output_root / design_name / "MPNN_outputs"
    root.mkdir(parents=True, exist_ok=True)
    engine = MPNNInferenceEngine(
        model_type=mpnn_settings["model_type"],
        is_legacy_weights=mpnn_settings["is_legacy_weights"],
        out_directory=str(root),
    )
    # Keys that are engine-level (not per-input) – skip them in input_dicts
    _ENGINE_KEYS = {"model_type", "is_legacy_weights"}
    for p in input_files:
        tag = strip_known_suffixes(p)
        arr = load_atom_array_from_file(p)
        out_dir = root / tag
        out_dir.mkdir(parents=True, exist_ok=True)
        # Build complete per-input dict; skip None values and engine-level keys
        input_dict = {"name": tag}
        for k, v in mpnn_settings.items():
            if k in _ENGINE_KEYS:
                continue
            if v is not None:
                input_dict[k] = v
        outs = engine.run(input_dicts=[input_dict], atom_arrays=[arr])
        for i, out in enumerate(outs):
            name_with_extension = f"{tag}_mpnn_{i:02d}.cif"
            out.write_structure(base_path=out_dir / name_with_extension)
            mpnn_files.append(out_dir / name_with_extension)
            rows.append({
                "source": tag,
                "sample": name_with_extension,
                "sequence": out.output_dict.get("designed_sequence", ""),
            })
    return pd.DataFrame(rows), mpnn_files

# This function is used to interactively browse through the generated structures.
def show_batch_browser(array_per_batch: list[list]):
    if not array_per_batch or not any(array_per_batch):
        info_box("No structures to browse", "Run the RFD3 step first, then come back to this viewer.")
        return

    batch_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=max(0, len(array_per_batch) - 1),
        description="Batch",
        continuous_update=False,
    )
    struct_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=0,
        description="Design",
        continuous_update=False,
    )

    # Use an Output widget to hold the py3Dmol viewer, making updates more robust
    viz_out = widgets.Output(layout={'border': '1px solid #ddd', 'min_height': '620px'})

    def update_design_slider():
        batch_number = batch_slider.value
        total = len(array_per_batch[batch_number])
        struct_slider.max = max(0, total - 1)
        if struct_slider.value > struct_slider.max:
            struct_slider.value = struct_slider.max

    def render(_=None):
        batch_idx = batch_slider.value
        # Ensure struct_slider value is within bounds for the current batch
        d_slider_max = max(0, len(array_per_batch[batch_idx]) - 1)
        design_idx = min(struct_slider.value, d_slider_max)

        if batch_idx >= len(array_per_batch) or not array_per_batch[batch_idx]:
            with viz_out:
                clear_output(wait=True)
                print("No designs found in this batch.")
            return

        with viz_out:
            clear_output(wait=True) # Clear previous content within this output widget
            v = view(
                array_per_batch[batch_idx][design_idx],
                show_hover=True,
                show_cartoon=True,
                show_surface=False,
                width=880,
                height=620,
            )
            display(HTML(v._make_html())) # Display py3Dmol viewer's HTML representation

    def on_struct_change(change):
        render()

    def on_batch_change(change):
        # Prevent cascade: temporarily remove struct observer while updating slider bounds
        struct_slider.unobserve(on_struct_change, names="value")
        update_design_slider()
        struct_slider.observe(on_struct_change, names="value")
        render()

    # Initialize sliders and display them along with the output area for the viewer
    update_design_slider()
    display(widgets.VBox([widgets.HBox([batch_slider, struct_slider]), viz_out]))

    # Initial render to populate the viz_out widget
    render()

    # Attach observers
    struct_slider.observe(on_struct_change, names="value")
    batch_slider.observe(on_batch_change, names="value")

def create_boltz_yaml_input(
    atom_array,
    output_dir: Path,
    design_name: str,
    boltz_settings: dict,
) -> Path:
    """Build a valid Boltz-2 input YAML from an AtomArray.

    Produces a YAML that matches the Boltz-2 specification:
      version / sequences / (optional) properties
    Ligand chains are detected as chains whose majority residues are not
    standard amino acids.  The first non-water residue name is used as the
    CCD code (works for most common ligands in the PDB).
    """
    _THREE_TO_ONE = {
        "ALA": "A", "CYS": "C", "ASP": "D", "GLU": "E", "PHE": "F",
        "GLY": "G", "HIS": "H", "ILE": "I", "LYS": "K", "LEU": "L",
        "MET": "M", "ASN": "N", "PRO": "P", "GLN": "Q", "ARG": "R",
        "SER": "S", "THR": "T", "VAL": "V", "TRP": "W", "TYR": "Y",
        "MSE": "M", "UNK": "X", "PYL": "O",
    }
    WATER_NAMES = {"HOH", "WAT", "H2O"}

    sequences = []
    properties = []

    chain_ids = list(dict.fromkeys(atom_array.chain_id))  # ordered unique
    for chain_id in chain_ids:
        chain_arr = atom_array[atom_array.chain_id == chain_id]
        res_starts = get_residue_starts(chain_arr)
        res_names = chain_arr.res_name[res_starts].tolist()

        non_water = [r for r in res_names if r not in WATER_NAMES]
        if not non_water:
            continue  # skip water-only chains

        protein_count = sum(1 for r in non_water if r in AA_CODES)
        is_protein = protein_count >= max(1, len(non_water) // 2)

        if is_protein:
            seq_str = "".join(
                _THREE_TO_ONE.get(r, "X") for r in non_water if r in AA_CODES or r in _THREE_TO_ONE
            )
            if seq_str:
                sequences.append({"protein": {"id": chain_id, "sequence": seq_str}})
        else:
            # Ligand: use the first non-water residue name as CCD code
            ccd_code = non_water[0]
            sequences.append({"ligand": {"id": chain_id, "ccd": ccd_code}})
            if boltz_settings.get("affinity") and boltz_settings.get("ligand_chain") == chain_id:
                properties.append({"affinity": {"binder": chain_id}})

    if not sequences:
        raise ValueError(f"No valid sequences found in atom array for {design_name}")

    boltz_yaml = {"version": 1, "sequences": sequences}
    if properties:
        boltz_yaml["properties"] = properties

    yaml_path = output_dir / f"{design_name}.yaml"
    with open(yaml_path, "w") as fh:
        yaml.dump(boltz_yaml, fh, default_flow_style=False, sort_keys=False)
    return yaml_path

def run_boltz_pipeline(
    design_name: str,
    input_files: list,
    boltz_settings: dict,
    output_root: Path,
) -> tuple:
    """Run Boltz-2 on each structure file and assemble a ranking DataFrame.

    Works with RFD3 or MPNN output files — whatever is passed in input_files.
    """
    if not input_files:
        info_box("Boltz-2 skipped", "No structure files provided to Boltz-2.")
        return pd.DataFrame(), None

    boltz_output_root = output_root / design_name / "boltz"
    boltz_output_root.mkdir(parents=True, exist_ok=True)
    ranking_rows = []

    for input_path in input_files:
        input_path = Path(input_path)
        tag = strip_known_suffixes(input_path)
        arr = load_atom_array_from_file(input_path)
        boltz_run_dir = boltz_output_root / tag
        boltz_run_dir.mkdir(parents=True, exist_ok=True)

        try:
            yaml_path = create_boltz_yaml_input(arr, boltz_run_dir, tag, boltz_settings)

            # Correct Boltz-2 CLI: positional `data` arg, --out_dir, --use_msa_server
            command = [
                "boltz", "predict", str(yaml_path),
                "--out_dir", str(boltz_run_dir),
                "--output_format", "pdb",
                "--diffusion_samples", str(boltz_settings["diffusion_samples"]),
                "--seed", str(boltz_settings["seed"]),
            ]
            if boltz_settings.get("msa_from_server"):
                command.append("--use_msa_server")

            run_command(command, f"Running Boltz-2 for {tag}")

            # Boltz writes results to {out_dir}/boltz_results_{yaml_stem}/
            boltz_results_dir = boltz_run_dir / f"boltz_results_{yaml_path.stem}"
            predictions_dir = boltz_results_dir / "predictions" / tag

            # Try ranking.json first (produced when affinity is enabled)
            ranking_json_path = boltz_results_dir / "ranking.json"
            if ranking_json_path.exists():
                with open(ranking_json_path) as fj:
                    for item in json.load(fj):
                        pred_name = item.get("design_name", tag)
                        pred_pdb = predictions_dir / f"{pred_name}_model_0.pdb"
                        ranking_rows.append({
                            "source": tag,
                            "source_file": str(input_path),
                            "sample": pred_name,
                            "sequence": item.get("seq", ""),
                            "boltz_p_conf": item.get("p_conf"),
                            "boltz_p_aff": item.get("p_aff"),
                            "boltz_aff": item.get("aff"),
                            "boltz_delta_sasa": item.get("delta_sasa"),
                            "boltz_polar_contacts": item.get("polar_contacts"),
                            "boltz_liability_score": item.get("liability_score"),
                            "prediction_pdb": str(pred_pdb) if pred_pdb.exists() else None,
                        })
            elif predictions_dir.exists():
                # Fall back to per-model confidence JSON
                for conf_json in sorted(predictions_dir.glob(f"confidence_{tag}_model_*.json")):
                    model_idx = conf_json.stem.rsplit("_", 1)[-1]
                    pred_pdb = predictions_dir / f"{tag}_model_{model_idx}.pdb"
                    with open(conf_json) as fj:
                        conf = json.load(fj)
                    ranking_rows.append({
                        "source": tag,
                        "source_file": str(input_path),
                        "sample": f"{tag}_model_{model_idx}",
                        "sequence": "",
                        "boltz_p_conf": conf.get("confidence_score"),
                        "boltz_p_aff": conf.get("iptm"),
                        "boltz_aff": None,
                        "boltz_delta_sasa": None,
                        "boltz_polar_contacts": None,
                        "boltz_liability_score": None,
                        "prediction_pdb": str(pred_pdb) if pred_pdb.exists() else None,
                    })
            else:
                print(f"Warning: Boltz-2 predictions not found for {tag} in {boltz_results_dir}.")

        except Exception as e:
            print(f"Error running Boltz-2 for {tag}: {e}")

    ranking_df = pd.DataFrame(ranking_rows)

    if not ranking_df.empty:
        # Normalize available numeric columns (fill missing with 0 for scoring)
        conf_col = ranking_df["boltz_p_conf"].fillna(0)
        aff_col  = ranking_df["boltz_p_aff"].fillna(0)

        max_conf = conf_col.max() or 1
        max_aff  = aff_col.max() or 1

        ranking_df["boltz_p_conf_norm"] = conf_col / max_conf
        ranking_df["boltz_p_aff_norm"]  = aff_col / max_aff

        score_cols = {"boltz_p_conf_norm": 0.6, "boltz_p_aff_norm": 0.4}
        for col, weight in score_cols.items():
            if col not in ranking_df.columns:
                ranking_df[col] = 0.0

        ranking_df["BoltzGen_Score"] = (
            ranking_df["boltz_p_conf_norm"] * 0.6
            + ranking_df["boltz_p_aff_norm"] * 0.4
        )
        ranking_df = ranking_df.sort_values("BoltzGen_Score", ascending=False).reset_index(drop=True)

        ranking_dir = output_root / design_name / "ranking"
        ranking_dir.mkdir(parents=True, exist_ok=True)
        ranking_path = ranking_dir / "boltz_ranking.xlsx"
        ranking_df.to_excel(ranking_path, index=False)
        return ranking_df, ranking_path

    return pd.DataFrame(), None



def superimpose_and_visualize(design_arr, boltz_pred_arr, design_label="RFD3 Design", pred_label="Boltz Prediction"):
    """Superimpose Boltz prediction onto the design and show both in py3Dmol.

    The design is shown in teal; the aligned Boltz prediction in light grey.
    CA RMSD is computed after alignment and shown in the info box.
    """
    try:
        import py3Dmol
        from biotite.structure import superimpose
        import tempfile
    except ImportError as e:
        info_box("Superposition skipped", f"Missing dependency: {e}")
        return

    def get_protein_ca(arr):
        return arr[(arr.atom_name == "CA") & np.isin(arr.res_name, list(AA_CODES))]

    design_ca = get_protein_ca(design_arr)
    pred_ca   = get_protein_ca(boltz_pred_arr)
    min_len   = min(len(design_ca), len(pred_ca))

    if min_len < 5:
        info_box("Superposition skipped",
                 f"Too few CA atoms to align ({min_len} matched). Need ≥ 5.")
        return

    try:
        aligned_pred_ca, transform = superimpose(design_ca[:min_len], pred_ca[:min_len])
        aligned_pred_full = transform.apply(boltz_pred_arr)
        aligned_pred_ca_full = get_protein_ca(aligned_pred_full)
        rmsd = float(np.sqrt(np.mean(np.sum(
            (design_ca[:min_len].coord - aligned_pred_ca_full[:min_len].coord) ** 2,
            axis=-1,
        ))))
    except Exception as e:
        info_box("Superposition warning", f"Alignment failed: {e}\nShowing raw structures side by side.")
        aligned_pred_full = boltz_pred_arr
        rmsd = float("nan")

    def arr_to_pdb_str(arr):
        pdb_obj = pdb.PDBFile()
        pdb_obj.set_structure(arr)
        with tempfile.NamedTemporaryFile(suffix=".pdb", delete=False) as tmp:
            tmp_path = tmp.name
        pdb_obj.write(tmp_path)
        with open(tmp_path) as fp:
            content = fp.read()
        Path(tmp_path).unlink()
        return content

    design_pdb_str = arr_to_pdb_str(design_arr)
    pred_pdb_str   = arr_to_pdb_str(aligned_pred_full)

    viewer = py3Dmol.view(width=880, height=620)
    viewer.addModel(design_pdb_str, "pdb")
    viewer.setStyle({"model": 0}, {"cartoon": {"color": "teal"}})
    viewer.addModel(pred_pdb_str, "pdb")
    viewer.setStyle({"model": 1}, {"cartoon": {"color": "lightgray"}})
    viewer.zoomTo()

    rmsd_str = f"{rmsd:.3f} Å" if rmsd == rmsd else "N/A"
    info_box(
        "Superposition: Design (teal) vs Boltz Prediction (grey)",
        f"{design_label}  ↔  {pred_label}\nCA RMSD = {rmsd_str}  |  Residues aligned = {min_len}",
    )
    display(HTML(viewer._make_html()))

def create_results_archive(design_name: str, output_root: Path) -> Path:
    archive_dir = DOWNLOAD_ROOT
    archive_dir.mkdir(parents=True, exist_ok=True)
    archive_path = archive_dir / f"{design_name}_results.zip"

    run_root = output_root / design_name

    temp_stage_dir = archive_dir / f"{design_name}_temp_stage"
    temp_stage_dir.mkdir(parents=True, exist_ok=True) # Ensure temp_stage_dir exists

    for subdir_name in ["RFD3_designs", "MPNN_outputs", "boltz", "ranking"]:
        source_path = run_root / subdir_name
        if source_path.exists():
            # shutil.copytree requires destination to not exist, unless dirs_exist_ok=True (Python 3.8+)
            # For broader compatibility, check and remove/create if needed, or copy contents instead of tree.
            # Simplified for Colab environment assuming newer Python and existing_ok
            destination_path = temp_stage_dir / subdir_name
            # Make sure parent directory exists for destination_path if copying sub-folders
            destination_path.parent.mkdir(parents=True, exist_ok=True)
            if destination_path.exists():
                shutil.rmtree(destination_path)
            shutil.copytree(source_path, destination_path)

    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(temp_stage_dir):
            for file in files:
                file_path = Path(root) / file
                arcname = file_path.relative_to(temp_stage_dir)
                zipf.write(file_path, arcname)

    shutil.rmtree(temp_stage_dir)

    return archive_path


info_box("Notebook helpers are ready", f"Workspace: {WORK_ROOT}")


info_box("Notebook helpers are ready", f"Workspace: {WORK_ROOT}")

In [ ]:
#@title 🧬 Load input structure
#@markdown Use the controls below to load a starting structure. If you choose upload mode, running the cell will open the Colab file picker.

structure_source = "Upload PDB/CIF" #@param ["Fetch from RCSB", "Upload PDB/CIF"]
rcsb_id = "1atz" #@param {type:"string"}
rcsb_format = "cif" #@param ["pdb", "cif"]

structure_array = None
structure_file_path = None
structure_label = None

if structure_source == "Upload PDB/CIF":
    if colab_files is None:
        raise ImportError("Upload mode requires Google Colab.")
    uploaded_files = colab_files.upload()
    if not uploaded_files:
        raise ValueError("No file was uploaded.")
    first_name = next(iter(uploaded_files))
    upload_dir = STRUCTURE_ROOT / "uploads"
    upload_dir.mkdir(parents=True, exist_ok=True)

    # Move the uploaded file from the default Colab root to the designated upload directory
    original_path = COLAB_ROOT / first_name
    desired_path = upload_dir / first_name
    shutil.move(original_path, desired_path)

    structure_file_path = desired_path # Update the path to the new location
    structure_label = first_name
    structure_array, structure_file_path = StructureFetcher.read_local(structure_file_path)
else:
    if not clean_text(rcsb_id):
        raise ValueError("Please provide an RCSB identifier.")
    structure_array, structure_file_path = StructureFetcher.from_rcsb(rcsb_id, file_format=rcsb_format, target_path=STRUCTURE_ROOT / "rcsb")
    structure_label = f"{rcsb_id.upper()} ({rcsb_format})"

PIPELINE_STATE["structure_array"] = structure_array
PIPELINE_STATE["structure_file_path"] = str(structure_file_path)
PIPELINE_STATE["structure_label"] = structure_label

info_box(
    "Input structure loaded",
    f"Source: {structure_source}\nFile: {structure_file_path}",
)
display(
    view(
        structure_array,
        show_hover=True,
        show_cartoon=True,
        show_surface=False,
        width=880,
        height=620,
    )
)

## Step 2. Configure and Run RFDiffusion3

The next three cells expose the main RFD3 controls.
Start with a simple design, then revisit the advanced settings only if you need finer control over motifs, surface classes, hydrogen bonds, symmetry, or the diffusion sampler.

In [ ]:
#@title ⚙️ RFD3 basic design settings
#@markdown These are the main settings most users need. Residue selections can be plain contig strings such as `A10-25,B3-7`. For atom-specific selections you can also enter a Python or JSON dictionary such as `{"IAI": "ALL", "A108": "N,CA,C,O"}`.

design_name = "interactive_rfd3_run" #@param {type:"string"}
seed = 42 #@param {type:"integer"}
diffusion_batch_size = 3 #@param {type:"slider", min:1, max:16, step:1}
num_batches = 1 #@param {type:"slider", min:1, max:10, step:1}
length = "180-220" #@param {type:"string"}
contig = "" #@param {type:"string"}
unindex = "" #@param {type:"string"}
ligand = "IAI" #@param {type:"string"}
select_fixed_atoms = '{"IAI": "ALL"}' #@param {type:"string"}
select_unfixed_sequence = "" #@param {type:"string"}
redesign_motif_sidechains = False #@param {type:"boolean"}
plddt_enhanced = True #@param {type:"boolean"}
is_non_loopy = True #@param {type:"boolean"}
partial_t = "" #@param {type:"string"}
skip_existing = True #@param {type:"boolean"}
dump_trajectories = False #@param {type:"boolean"}
align_trajectory_structures = False #@param {type:"boolean"}
dump_prediction_metadata_json = True #@param {type:"boolean"}
output_full_json = True #@param {type:"boolean"}
prevalidate_inputs = True #@param {type:"boolean"}
low_memory_mode = False #@param {type:"boolean"}

In [ ]:
#@title 🧪 Advanced RFD3 conditioning and diffusion settings
#@markdown Use these controls when you need buried or exposed residues, explicit hydrogen-bond annotations, hotspots, symmetry, or direct control over the diffusion sampler. Leave fields empty when they are not needed.

select_buried = "" #@param {type:"string"}
select_partially_buried = "" #@param {type:"string"}
select_exposed = "" #@param {type:"string"}
select_hbond_acceptor = "" #@param {type:"string"}
select_hbond_donor = "" #@param {type:"string"}
select_hotspots = "" #@param {type:"string"}
infer_ori_strategy = "None" #@param ["None", "com", "hotspots"]
ori_token = "" #@param {type:"string"}
symmetry_id = "None" #@param ["None", "C2", "C3", "C4", "C5", "C6", "D2", "D3", "D4", "D5", "D6"]
is_unsym_motif = "" #@param {type:"string"}
is_symmetric_motif = True #@param {type:"boolean"}
cleanup_guideposts = True #@param {type:"boolean"}
cleanup_virtual_atoms = True #@param {type:"boolean"}
read_sequence_from_sequence_head = True #@param {type:"boolean"}
global_prefix = "" #@param {type:"string"}
diffusion_kind = "default" #@param ["default", "symmetry"]
num_timesteps = 200 #@param {type:"integer"}
min_t = 0 #@param {type:"integer"}
max_t = 1 #@param {type:"number"}
sigma_data = 16 #@param {type:"number"}
s_min = 0.0004 #@param {type:"number"}
s_max = 160 #@param {type:"number"}
p = 7 #@param {type:"number"}
gamma_0 = 0.6 #@param {type:"number"}
gamma_min = 1.0 #@param {type:"number"}
noise_scale = 1.003 #@param {type:"number"}
step_scale = 1.5 #@param {type:"number"}
solver = "af3" #@param ["af3"]
center_option = "all" #@param ["all", "motif", "diffuse"]
s_trans = 1.0 #@param {type:"number"}
s_jitter_origin = 0.0 #@param {type:"number"}
fraction_of_steps_to_fix_motif = 0.0 #@param {type:"number"}
skip_few_diffusion_steps = False #@param {type:"boolean"}
allow_realignment = False #@param {type:"boolean"}
insert_motif_at_end = True #@param {type:"boolean"}
use_classifier_free_guidance = False #@param {type:"boolean"}
cfg_scale = 2.0 #@param {type:"number"}
cfg_t_max = "" #@param {type:"string"}

In [ ]:
#@title ▶️ Preview the specification and run RFD3
#@markdown This cell builds the validated DesignInputSpecification, shows the conditioned input structure, and optionally launches RFDiffusion3.

run_rfd3_now = True #@param {type:"boolean"}

if "structure_array" not in PIPELINE_STATE:
    raise ValueError("Load an input structure before running RFD3.")

seed_everything(seed)

symmetry_config = build_symmetry_config(symmetry_id, is_unsym_motif, is_symmetric_motif)
if symmetry_config is not None and diffusion_kind == "default":
    diffusion_kind = "symmetry"

sampler_kwargs = {
    "kind": diffusion_kind,
    "num_timesteps": int(num_timesteps),
    "min_t": int(min_t),
    "max_t": float(max_t),
    "sigma_data": float(sigma_data),
    "s_min": float(s_min),
    "s_max": float(s_max),
    "p": float(p),
    "gamma_0": float(gamma_0),
    "gamma_min": float(gamma_min),
    "noise_scale": float(noise_scale),
    "step_scale": float(step_scale),
    "solver": solver,
    "center_option": center_option,
    "s_trans": float(s_trans),
    "s_jitter_origin": float(s_jitter_origin),
    "fraction_of_steps_to_fix_motif": float(fraction_of_steps_to_fix_motif),
    "skip_few_diffusion_steps": bool(skip_few_diffusion_steps),
    "allow_realignment": bool(allow_realignment),
    "insert_motif_at_end": bool(insert_motif_at_end),
    "use_classifier_free_guidance": bool(use_classifier_free_guidance),
    "cfg_scale": float(cfg_scale),
}
if clean_text(cfg_t_max):
    sampler_kwargs["cfg_t_max"] = float(cfg_t_max)

input_kwargs = {
    "atom_array_input": PIPELINE_STATE["structure_array"],
    "input": None,
    "contig": parse_literal_or_text(contig),
    "unindex": parse_literal_or_text(unindex),
    "length": clean_text(length),
    "ligand": clean_text(ligand),
    "select_fixed_atoms": parse_literal_or_text(select_fixed_atoms),
    "select_unfixed_sequence": parse_literal_or_text(select_unfixed_sequence),
    "select_buried": parse_literal_or_text(select_buried),
    "select_partially_buried": parse_literal_or_text(select_partially_buried),
    "select_exposed": parse_literal_or_text(select_exposed),
    "select_hbond_acceptor": parse_literal_or_text(select_hbond_acceptor),
    "select_hbond_donor": parse_literal_or_text(select_hbond_donor),
    "select_hotspots": parse_literal_or_text(select_hotspots),
    "redesign_motif_sidechains": bool(redesign_motif_sidechains),
    "symmetry": symmetry_config,
    "ori_token": parse_float_vector(ori_token),
    "infer_ori_strategy": None if infer_ori_strategy == "None" else infer_ori_strategy,
    "plddt_enhanced": bool(plddt_enhanced),
    "is_non_loopy": bool(is_non_loopy),
    "partial_t": parse_literal_or_text(partial_t),
}
# Ensure 'input' key is always present, even if its value is None, to satisfy rfd3's internal validator.
input_kwargs = {key: value for key, value in input_kwargs.items() if value is not None or key == "input"}
design_spec = DesignInputSpecification(**input_kwargs)
preview = design_spec.build()
preview_array = preview[0] if isinstance(preview, (tuple, list)) else preview

rfd3_config = {
    "diffusion_batch_size": int(diffusion_batch_size),
    "skip_existing": bool(skip_existing),
    "dump_prediction_metadata_json": bool(dump_prediction_metadata_json),
    "dump_trajectories": bool(dump_trajectories),
    "align_trajectory_structures": bool(align_trajectory_structures),
    "output_full_json": bool(output_full_json),
    "cleanup_guideposts": bool(cleanup_guideposts),
    "cleanup_virtual_atoms": bool(cleanup_virtual_atoms),
    "read_sequence_from_sequence_head": bool(read_sequence_from_sequence_head),
    "prevalidate_inputs": bool(prevalidate_inputs),
    "low_memory_mode": bool(low_memory_mode),
    "global_prefix": clean_text(global_prefix),
    "seed": int(seed),
    "inference_sampler": sampler_kwargs,
}
rfd3_config = {key: value for key, value in rfd3_config.items() if value is not None}

PIPELINE_STATE["design_name"] = design_name
PIPELINE_STATE["design_spec"] = design_spec
PIPELINE_STATE["rfd3_config"] = rfd3_config
PIPELINE_STATE["rfd3_output_root"] = OUTPUT_ROOT
PIPELINE_STATE["sampler_kwargs"] = sampler_kwargs

info_box(
    "Design specification ready",
    f"Design name: {design_name}\nDiffusion batch size: {diffusion_batch_size}\nNumber of batches: {num_batches}",
)
display(
    view(
        preview_array,
        show_hover=True,
        show_cartoon=True,
        show_surface=False,
        width=880,
        height=620,
    )
)

if run_rfd3_now:
    arrays_by_batch, rfd3_files = run_rfd3_batches(
        design_name=design_name,
        rfd3_config=rfd3_config,
        design_spec=design_spec,
        num_batches=int(num_batches),
        output_root=OUTPUT_ROOT,
    )
    PIPELINE_STATE["rfd3_arrays_by_batch"] = arrays_by_batch
    PIPELINE_STATE["rfd3_files"] = [str(path) for path in rfd3_files]
    info_box(
        "RFD3 completed",
        f"Generated {len(rfd3_files)} structures across {num_batches} batch(es). Results were saved under {OUTPUT_ROOT / design_name}.",
    )
    show_batch_browser(arrays_by_batch)
else:
    info_box("Preview only", "RFD3 was not executed because the run switch is off.")

## Step 3. Design Sequences with ProteinMPNN or LigandMPNN

This stage takes each RFD3 backbone and proposes amino-acid sequences.
Use **LigandMPNN** when your design contains a ligand or explicit non-protein context, and **ProteinMPNN** for standard protein-only designs.

In [ ]:
#@title 🧪 Run ProteinMPNN or LigandMPNN
#@markdown Leave advanced fields empty unless you need residue-level restrictions, biasing, omission rules, or symmetry coupling.

mpnn_model_type = "ligand_mpnn" #@param ["ligand_mpnn", "protein_mpnn"]
mpnn_batch_size = 8 #@param {type:"slider", min:1, max:32, step:1}
mpnn_number_of_batches = 1 #@param {type:"slider", min:1, max:8, step:1}
mpnn_temperature = 0.1 #@param {type:"number"}
mpnn_seed = 42 #@param {type:"integer"}
mpnn_remove_waters = True #@param {type:"boolean"}
mpnn_is_legacy_weights = True #@param {type:"boolean"}
mpnn_structure_noise = 0.0 #@param {type:"number"}
mpnn_decode_type = "auto_regressive" #@param ["auto_regressive", "teacher_forcing"]
mpnn_causality_pattern = "auto_regressive" #@param ["auto_regressive", "unconditional", "conditional", "conditional_minus_self"]
mpnn_initialize_with_ground_truth = False #@param {type:"boolean"}
mpnn_atomize_side_chains = False #@param {type:"boolean"}
mpnn_remove_ccds = "" #@param {type:"string"}
mpnn_fixed_residues = "" #@param {type:"string"}
mpnn_designed_residues = "" #@param {type:"string"}
mpnn_fixed_chains = "" #@param {type:"string"}
mpnn_designed_chains = "" #@param {type:"string"}
mpnn_bias = "" #@param {type:"string"}
mpnn_bias_per_residue = "" #@param {type:"string"}
mpnn_omit = "UNK" #@param {type:"string"}
mpnn_omit_per_residue = "" #@param {type:"string"}
mpnn_pair_bias = "" #@param {type:"string"}
mpnn_pair_bias_per_residue_pair = "" #@param {type:"string"}
mpnn_temperature_per_residue = "" #@param {type:"string"}
mpnn_symmetry_residues = "" #@param {type:"string"}
mpnn_symmetry_residues_weights = "" #@param {type:"string"}
mpnn_homo_oligomer_chains = "" #@param {type:"string"}
run_mpnn_now = True #@param {type:"boolean"}

# Use MPNN outputs > RFD3 outputs > input structure, in that priority order
_mpnn_input_files, _mpnn_source = get_current_structure_files()

mpnn_settings = {
    "model_type": mpnn_model_type,
    "is_legacy_weights": bool(mpnn_is_legacy_weights),
    "seed": int(mpnn_seed),
    "batch_size": int(mpnn_batch_size),
    "number_of_batches": int(mpnn_number_of_batches),
    "remove_ccds": parse_csv_list(mpnn_remove_ccds) or [],
    "remove_waters": bool(mpnn_remove_waters),
    "structure_noise": float(mpnn_structure_noise),
    "decode_type": mpnn_decode_type,
    "causality_pattern": mpnn_causality_pattern,
    "initialize_sequence_embedding_with_ground_truth": bool(mpnn_initialize_with_ground_truth),
    "atomize_side_chains": bool(mpnn_atomize_side_chains),
    "fixed_residues": parse_csv_list(mpnn_fixed_residues),
    "designed_residues": parse_csv_list(mpnn_designed_residues),
    "fixed_chains": parse_csv_list(mpnn_fixed_chains),
    "designed_chains": parse_csv_list(mpnn_designed_chains),
    "bias": parse_literal_or_text(mpnn_bias),
    "bias_per_residue": parse_literal_or_text(mpnn_bias_per_residue),
    "omit": parse_csv_list(mpnn_omit) or ["UNK"],
    "omit_per_residue": parse_literal_or_text(mpnn_omit_per_residue),
    "pair_bias": parse_literal_or_text(mpnn_pair_bias),
    "pair_bias_per_residue_pair": parse_literal_or_text(mpnn_pair_bias_per_residue_pair),
    "temperature": float(mpnn_temperature),
    "temperature_per_residue": parse_literal_or_text(mpnn_temperature_per_residue),
    "symmetry_residues": parse_literal_or_text(mpnn_symmetry_residues),
    "symmetry_residues_weights": parse_literal_or_text(mpnn_symmetry_residues_weights),
    "homo_oligomer_chains": parse_literal_or_text(mpnn_homo_oligomer_chains),
}

if run_mpnn_now:
    sequence_df, mpnn_cif_files = run_mpnn_pipeline(
        design_name=PIPELINE_STATE.get("design_name", "run"),
        input_files=_mpnn_input_files,
        mpnn_settings=mpnn_settings,
        output_root=OUTPUT_ROOT,
    )
    PIPELINE_STATE["mpnn_sequence_df"] = sequence_df
    PIPELINE_STATE["mpnn_cif_files"] = [str(path) for path in mpnn_cif_files]
    info_box(
        "MPNN completed",
        f"Generated {len(sequence_df)} sequence proposals from {len(PIPELINE_STATE['rfd3_files'])} RFD3 backbone(s).",
    )
    display(sequence_df.head(25))
else:
    info_box("MPNN skipped", "The run switch is off. No sequence design was executed.")


┌──────────────────────────────────────────┐
│ MPNN completed
├──────────────────────────────────────────┤
│ Generated 24 sequence proposals from 3 RFD3 backbone(s).
└──────────────────────────────────────────┘



,source,sample,sequence
0,backbone_0_0_model_0,backbone_0_0_model_0_mpnn_00.cif,HSPEEVRRRAMEVLRELIRAEEASARICTEAVETLSSNDLSTDEQI...
1,backbone_0_0_model_0,backbone_0_0_model_0_mpnn_01.cif,HSPEEVRQAAIEVLELLIEAETESARIAAEAVERLESEDLSTEERI...
2,backbone_0_0_model_0,backbone_0_0_model_0_mpnn_02.cif,PSPEEVRAAAIEVLRLLIEAERKSAEIARKAVEILSSNDLSTEEQI...
3,backbone_0_0_model_0,backbone_0_0_model_0_mpnn_03.cif,ASPEEVRREAIRNLELLIEAERASAEICRRAVETLESNDLSTDEVI...
4,backbone_0_0_model_0,backbone_0_0_model_0_mpnn_04.cif,HSPEEIRAAAIEILEELIRAERESAQIAAEAVETLSSRDLSTEEQI...
5,backbone_0_0_model_0,backbone_0_0_model_0_mpnn_05.cif,ASPEEVRAAAARVLRKLIEAEEKSAEVLSKAVETLESNDLSTDEQI...
6,backbone_0_0_model_0,backbone_0_0_model_0_mpnn_06.cif,HSPAEVRARAMENLRLLIEAERESARVCAEAVRTLEENDLSTDEAI...
7,backbone_0_0_model_0,backbone_0_0_model_0_mpnn_07.cif,PSPEEVRAAAIEVLEALIEAEERSARICRRAVEILESNDLSTEEQI...
8,backbone_0_0_model_1,backbone_0_0_model_1_mpnn_00.cif,MMLLLAYLIGLLAALVNALGCVLAEKEKGLNAVDVMAVIGNVVMAV...
9,backbone_0_0_model_1,backbone_0_0_model_1_mpnn_01.cif,MMLLLALAIGVLAALVNAAGCVAAERDRGLHAVDVMAVIGNVVIAV...


## Step 4. Predict with Boltz-2 and Rank the Results

Boltz-2 is used here as an external prediction engine after sequence design.
The notebook converts each MPNN structure into Boltz input YAML, runs prediction, then ranks the results with confidence, affinity, interface burial, polar contacts, and sequence-liability terms.

In [ ]:
#@title 🔮 Run Boltz-2 and build the ranking table
#@markdown Boltz-2 can take time. Start with a small number of diffusion samples while validating your pipeline, then increase the value for final ranking runs.

boltz_diffusion_samples = 3 #@param {type:"slider", min:1, max:16, step:1}
boltz_seed = 42 #@param {type:"integer"}
boltz_affinity = True #@param {type:"boolean"}
boltz_ligand_chain = "B" #@param {type:"string"}
boltz_use_msa_server = True #@param {type:"boolean"}
run_boltz_now = True #@param {type:"boolean"}

# Use MPNN outputs > RFD3 outputs > input structure, in that priority order
_boltz_input_files, _boltz_source = get_current_structure_files()

boltz_settings = {
    "diffusion_samples": int(boltz_diffusion_samples),
    "seed": int(boltz_seed),
    "affinity": bool(boltz_affinity),
    "ligand_chain": clean_text(boltz_ligand_chain) or "B",
    "msa_from_server": bool(boltz_use_msa_server),
}

if run_boltz_now:
    ranking_df, ranking_path = run_boltz_pipeline(
        design_name=PIPELINE_STATE.get("design_name", "run"),
        input_files=_boltz_input_files,
        boltz_settings=boltz_settings,
        output_root=OUTPUT_ROOT,
    )
    PIPELINE_STATE["ranking_df"] = ranking_df
    PIPELINE_STATE["ranking_path"] = str(ranking_path) if ranking_path is not None else None

    if ranking_df.empty:
        info_box("Boltz-2 finished without ranking rows", "No ranking table could be assembled from the prediction outputs.")
    else:
        info_box(
            "Boltz-2 ranking completed",
            f"Saved ranking workbook to {ranking_path}. Showing the top 10 rows below.",
        )
        display(ranking_df.head(10))

        top_prediction_path = ranking_df.iloc[0].get("prediction_pdb")
        if top_prediction_path and Path(top_prediction_path).exists():
            info_box("Top-ranked structure preview", f"Previewing {top_prediction_path}.")
            display(
                view(
                    load_atom_array_from_file(top_prediction_path),
                    show_hover=True,
                    show_cartoon=True,
                    show_surface=False,
                    width=880,
                    height=620,
                )
            )
        # --- Superposition visualization (Issue 6) ---
        top_row = ranking_df.iloc[0]
        top_pred_pdb = top_row.get("prediction_pdb")
        top_source_file = top_row.get("source_file")
        if top_pred_pdb and Path(top_pred_pdb).exists() and top_source_file and Path(top_source_file).exists():
            try:
                design_arr_for_super = load_atom_array_from_file(top_source_file)
                boltz_arr_for_super = load_atom_array_from_file(top_pred_pdb)
                superimpose_and_visualize(
                    design_arr_for_super,
                    boltz_arr_for_super,
                    design_label=Path(top_source_file).stem,
                    pred_label=str(top_row.get("sample", "Boltz Prediction")),
                )
            except Exception as _e:
                print(f"Superposition visualization error: {_e}")

else:
    info_box("Boltz-2 skipped", "The run switch is off. No Boltz prediction was executed.")

In [ ]:
#@title 📥 Download outputs from Colab
#@markdown Use this cell after at least one design stage has finished. It can download the complete run as a ZIP file and, if available, the ranking workbook separately.

download_results_zip = True #@param {type:"boolean"}
download_ranking_workbook = True #@param {type:"boolean"}

if "design_name" not in PIPELINE_STATE:
    raise ValueError("Run at least the RFD3 setup cell before downloading results.")

run_root = OUTPUT_ROOT / PIPELINE_STATE["design_name"]
run_root.mkdir(parents=True, exist_ok=True)
archive_path = create_results_archive(PIPELINE_STATE["design_name"], OUTPUT_ROOT)
ranking_path = PIPELINE_STATE.get("ranking_path")

info_box(
    "Download package ready",
    f"Archive: {archive_path}\nRun directory: {run_root}",
)

if colab_files is None:
    print("Direct browser downloads are available only in Google Colab. You can still use the paths above.")
else:
    if download_results_zip:
        colab_files.download(str(archive_path))
    if download_ranking_workbook and ranking_path:
        colab_files.download(str(ranking_path))
    elif download_ranking_workbook and not ranking_path:
        print("No ranking workbook is available yet. Run the Boltz-2 cell first.")

## Output locations

After a run the notebook writes outputs into the `outputs` directory under the Colab root. Use the design name you provided when running the pipeline.

- `/content/outputs/{design_name}/batch_XX/RFD3_designs/` — RFD3 backbones
- `/content/outputs/{design_name}/batch_XX/MPNN_outputs/` — designed sequences and CIF structures
- `/content/outputs/{design_name}/boltz/` — Boltz YAML files and predictions
- `/content/outputs/{design_name}/ranking/boltz_ranking.xlsx` — final ranking workbook

Use the Download cell to package and download the selected output folders.

## Practical Guide to the Controls in This Notebook



This appendix explains what the main controls do, when to change them, and which settings are best left alone unless you are deliberately reproducing a protocol.



The workflow in this notebook has four stages:



1. Load a starting structure.

2. Generate backbone designs with RFdiffusion3 (RFD3).

3. Design amino-acid sequences with ProteinMPNN or LigandMPNN.

4. Optionally run Boltz-2 to predict and rank the final designs.



If you are new to this workflow, change only a few settings at a time. A good first run uses one structure, one batch, a small `diffusion_batch_size`, default advanced settings, and a small Boltz sample count.



> **Known notebook caveats**

>

> - The main workflow is usable, but a few advanced notebook features still need cleanup.

> - `generate_yamls_only` is exposed in the interface, but it is not yet fully wired to skip Boltz prediction.

> - Upload mode and Boltz YAML export should be treated as features that may need validation if you rely on them heavily.

> - The output path note earlier in the notebook places MPNN outputs under `batch_XX`, but the current implementation writes them under `outputs/{design_name}/MPNN_outputs/`.



## Start Here: Which Settings Matter For My Task?



Use this quick guide before diving into the full parameter reference.



- **Small-molecule binder design**: focus on `ligand`, `length`, `select_fixed_atoms`, `select_buried`, and `select_exposed`.

- **Protein binder design**: focus on `contig`, `select_hotspots`, `infer_ori_strategy`, and `is_non_loopy`.

- **Enzyme or active-site scaffolding**: focus on `unindex`, `select_fixed_atoms`, `length`, and sometimes `redesign_motif_sidechains`.

- **DNA or RNA binder design**: focus on `contig`, `ori_token`, `select_fixed_atoms`, and optional hydrogen-bond constraints.

- **Symmetric assemblies**: focus on `symmetry_id`, `is_unsym_motif`, `is_symmetric_motif`, and `diffusion_kind="symmetry"`.

- **Redesigning an existing fold instead of starting from scratch**: focus on `partial_t` and a careful definition of which atoms stay fixed.



## Safe Defaults For First Runs



These defaults are good starting points for many exploratory runs.



- Keep `num_batches = 1` while debugging inputs.

- Keep `diffusion_batch_size` small, such as `1` to `4`.

- Leave most advanced diffusion settings unchanged unless you are reproducing a known recipe.

- Keep `prevalidate_inputs = True` so bad selections fail early.

- Keep `is_non_loopy = True` unless you want more flexible or loop-heavy backbones.

- Keep `low_memory_mode = False` unless memory is tight.

- For Boltz, start with a small `boltz_diffusion_samples` value such as `1` to `3`.



## The Most Important Idea: How You Describe The Design



RFD3 needs a description of what to copy from the input structure and what to create from scratch. The most important controls for that are `length`, `contig`, `unindex`, `ligand`, and the selection fields.



### `length`



`length` tells RFD3 how long the designed system should be.



- Use a single value such as `180` when you want one exact total length.

- Use a range such as `180-220` when you want RFD3 to explore several lengths.

- In nucleic-acid or multi-component cases, this is the total polymer length, not just the length of the newly designed protein chain.

- Do **not** use `length` together with `partial_t`; partial diffusion uses the input structure length.



### `contig`



`contig` is the main composition string. It tells RFD3 which parts come from the input structure and which parts should be generated.



Rules of thumb:



- A residue with a chain label such as `A10-25` means "use these residues from the input structure".

- A number or range without a chain label such as `50` or `40-60` means "design a new segment of this length".

- `/0` means a chain break.

- Commas separate pieces of the design.



Examples:



- `A10-25,50-80` means "copy residues A10 to A25, then design a connected segment that is 50 to 80 residues long".

- `40-60,/0,E6-155` means "design a new chain of 40 to 60 residues, then keep target residues E6 to E155 as a separate chain".

- `A1-10,/0,B15-24,/0,120-130` means "keep two nucleic-acid segments as fixed chains and design a separate protein chain of 120 to 130 residues".



### `unindex`



`unindex` is for motif residues that matter chemically, but whose exact sequence position is not fixed in advance.



This is especially useful in enzyme design or active-site scaffolding.



In plain language: use `unindex` when you know certain residues or ligand-contact atoms must survive, but you do not want to force them to stay at one exact residue number in the final design.



Example:



- `A108,A139,A152,A156` means these motif pieces should be present, but the model can infer where they belong in sequence.



### `ligand`



`ligand` tells RFD3 which non-protein component from the input structure should be treated as important.



- Usually this is a three-letter CCD name such as `IAI`.

- Multiple ligands can be given as a comma-separated string such as `NAI,ACT`.

- Use this whenever the design should respond to a ligand, cofactor, metal, or other non-protein context.



## How Selections Work



Several fields use the same idea: you select residues or atoms and tell the model what role they should play.



These fields include:



- `select_fixed_atoms`

- `select_unfixed_sequence`

- `select_buried`

- `select_partially_buried`

- `select_exposed`

- `select_hbond_acceptor`

- `select_hbond_donor`

- `select_hotspots`



A selection can often be written in more than one way.



- A simple contig-like string such as `A10-25,B3-7`

- A dictionary such as `{"IAI": "ALL", "A108": "N,CA,C,O"}`

- In some cases a boolean such as `True` or `False`



### `select_fixed_atoms`



This is one of the most important controls.



In plain language: it says which atoms should stay in the same 3D position instead of being moved by the model.



Common patterns:



- `True`: fix all atoms pulled from the input.

- `False`: allow all pulled atoms to move.

- `{"IAI": "ALL"}`: keep all atoms of ligand `IAI` fixed.

- `{"A108": "N,CA,C,O"}`: keep only specific backbone atoms fixed.

- `BKBN`: shorthand for backbone atoms.

- `TIP`: shorthand for tip atoms used in motif-style conditioning.

- `ALL`: all atoms in that selected residue or component.

- `""` as the atom list: intentionally select none of the atoms for fixing.



### `select_unfixed_sequence`



This says where the amino-acid identity is allowed to change.



In plain language: fixed coordinates and fixed sequence are different things. A residue can stay in place in 3D while still being allowed to mutate.



Example:



- `A20-35` means residues A20 to A35 can change amino-acid identity.



### `select_buried`, `select_partially_buried`, `select_exposed`



These control surface accessibility.



In plain language:



- `select_buried`: atoms or residues that should end up tucked into the design.

- `select_exposed`: atoms or residues that should stay accessible to solvent.

- `select_partially_buried`: intermediate accessibility.



These are especially useful for small-molecule binding, where one face of the ligand should be buried and another face should remain exposed.



### `select_hbond_acceptor` and `select_hbond_donor`



These add atom-level hydrogen-bond conditioning.



Use them when you want the design to respect a known hydrogen-bond pattern, especially in nucleic-acid or catalytic designs.



Example:



- `{"C16": "N7,O6"}` marks those atoms as hydrogen-bond acceptors.



### `select_hotspots`



This marks residues or atoms on the target that the new design should contact.



In plain language: these are the places you most want the binder to engage.



This is one of the most important controls for protein binder design.



## RFD3 Basic Design Settings



These are the fields in the "RFD3 basic design settings" cell.



### `design_name`



This is only a run label.



- It determines output folder names and archive names.

- It does **not** affect the science or model behavior.



### `seed`



This controls random-number generation.



- Reusing the same seed with the same settings can help reproducibility.

- Changing the seed can give different designs even when everything else is unchanged.



### `diffusion_batch_size`



This is how many RFD3 designs are generated in one sampling batch.



- Larger values produce more structures per run.

- Larger values also use more memory.

- For symmetry jobs, small values are strongly recommended.



### `num_batches`



This repeats the RFD3 generation process multiple times.



- Total number of backbone designs is roughly `num_batches × diffusion_batch_size`.

- Use `1` while testing inputs.



### `redesign_motif_sidechains`



Use this when you want motif side chains to be redesigned instead of copied exactly.



This is useful when a motif backbone must stay but side-chain identities should be reconsidered.



### `plddt_enhanced`



This enables a confidence-related enhancement used by the model.



For most users: leave this on.



### `is_non_loopy`



This encourages more structured designs with fewer floppy loops.



- Often recommended for binder design.

- If you want more flexible or loop-rich geometries, you may choose to turn it off.



### `partial_t`



This turns the run into **partial diffusion** instead of fully de novo generation.



In plain language: the model starts from a known structure and perturbs it by a chosen amount, instead of inventing a backbone from scratch.



Practical guidance:



- Typical useful values are around `5.0` to `15.0` Å.

- Smaller values stay closer to the starting structure.

- Larger values allow more structural change.

- Do not combine this with `length`.



### `skip_existing`



If outputs already exist, this can skip rerunning the same designs.



This is useful for resuming interrupted work, but it can also confuse you if you forget it is on and expect new outputs.



### `dump_trajectories`



This saves intermediate diffusion trajectory files.



Useful for debugging or visualization, but usually not needed for routine runs.



### `align_trajectory_structures`



This aligns saved trajectory structures for easier inspection.



Usually leave this off unless you are studying the trajectory itself.



### `dump_prediction_metadata_json`



This saves metadata JSON files alongside the structures.



Useful for traceability and debugging. Usually leave it on.



### `output_full_json`



This controls how much JSON metadata is saved.



Usually leave it on unless file size becomes a concern.



### `prevalidate_inputs`



This checks the design specification before a long run begins.



For most users: keep this on.



### `low_memory_mode`



This switches RFD3 into a more memory-efficient mode.



- Use it when GPU memory is insufficient.

- Expect slower runs.



## RFD3 Advanced Conditioning Settings



These fields change the behavior of the design task itself.



### `infer_ori_strategy`



This tells RFD3 how to place the designed region relative to the input structure.



Options:



- `None`: do not infer an origin automatically.

- `com`: use the center of mass of the input structure.

- `hotspots`: infer placement from the hotspot region.



For protein binder design, the repository docs specifically recommend `hotspots`.



### `ori_token`



This is an explicit `[x, y, z]` position for the design origin.



In plain language: it helps steer where the new design is placed in 3D space relative to the input.



Use it when you know the approximate desired location of the new designed region.



### Symmetry Controls



These fields are only important for symmetric designs.



- `symmetry_id`: the symmetry group, such as `C2`, `C3`, `D2`, or `D4`.

- `is_unsym_motif`: motif or component names that should **not** be symmetrized, such as DNA strands or a single ligand.

- `is_symmetric_motif`: whether the input motif is already symmetric.

- `diffusion_kind`: use `symmetry` for symmetry-aware diffusion.



Practical guidance:



- Use symmetry only when the design task truly requires it.

- Start with `diffusion_batch_size = 1` for symmetry jobs.

- Expect higher memory use.



## Advanced Diffusion Sampler Settings



These controls affect how the diffusion process behaves. Most users should leave them at their defaults unless reproducing a known protocol.



### Usually understandable and sometimes useful



- `num_timesteps`: number of diffusion steps. More steps can increase runtime.

- `noise_scale`: how much random noise is injected during sampling. Lower values usually make results more conservative.

- `step_scale`: scales the step size during sampling. The docs note that larger values can give less diverse but more designable outputs.

- `gamma_0`: influences diversity. Lower values are often described as giving lower-temperature, more designable outputs.

- `center_option`: how coordinates are centered during inference. Options are `all`, `motif`, and `diffuse`.

- `s_trans`: translational augmentation scale.

- `s_jitter_origin`: how much random jitter to add to the origin.

- `allow_realignment`: whether motif-based realignment is allowed during sampling.



### Expert-only controls



These are best left unchanged unless you know exactly why you are changing them.



- `min_t`

- `max_t`

- `sigma_data`

- `s_min`

- `s_max`

- `p`

- `gamma_min`

- `fraction_of_steps_to_fix_motif`

- `skip_few_diffusion_steps`

- `insert_motif_at_end`

- `solver`



### Classifier-Free Guidance Controls



These are advanced steering options.



- `use_classifier_free_guidance`: turn guidance on.

- `cfg_scale`: how strongly guidance influences sampling.

- `cfg_t_max`: the latest timepoint up to which guidance is applied.



For most users: leave guidance off unless you are explicitly experimenting with conditional steering.



### Debug-Oriented Output Controls



These are useful mainly when diagnosing tricky setups.



- `cleanup_guideposts`: if turned off, keeps extra guidepost information that can help debug unindexed motif setups.

- `cleanup_virtual_atoms`: if turned off, keeps virtual atoms used internally during diffusion. This is mainly for debugging and may make downstream tools less happy.

- `read_sequence_from_sequence_head`: whether saved sequence annotations come from the model sequence head. Usually leave this on.

- `global_prefix`: adds a custom filename prefix to outputs.



## Sequence Design With ProteinMPNN Or LigandMPNN



This stage takes each RFD3 backbone and proposes amino-acid sequences that fit it.



### `mpnn_model_type`



Choose the sequence-design model.



- `protein_mpnn`: use this for protein-only contexts.

- `ligand_mpnn`: use this when ligands or non-protein context matter.



If your design contains a small molecule, cofactor, or similar context, `ligand_mpnn` is usually the better choice.



### `mpnn_batch_size` and `mpnn_number_of_batches`



These determine how many sequence proposals are generated.



- `mpnn_batch_size`: number of samples drawn per batch.

- `mpnn_number_of_batches`: how many such batches to run.



More samples mean more diversity, but also more outputs to inspect.



### `mpnn_temperature`



This controls sampling diversity.



- Lower temperature gives more conservative sequences.

- Higher temperature gives more diverse and sometimes riskier sequences.



### `mpnn_seed`



Controls reproducibility for sequence sampling.



### `mpnn_remove_waters` and `mpnn_remove_ccds`



These clean the structure context before sequence design.



- `mpnn_remove_waters = True` is often sensible.

- `mpnn_remove_ccds` removes named chemical components you do not want considered.



### `mpnn_structure_noise`



Adds coordinate noise in Å before sequence design.



Most users should leave this at `0.0` unless deliberately testing robustness.



### `mpnn_decode_type`



This controls how residues are decoded.



- `auto_regressive`: each prediction uses previously predicted residues. This is the standard inference setting.

- `teacher_forcing`: each prediction sees the original input sequence at previous positions.



For most users: keep `auto_regressive`.



### `mpnn_causality_pattern`



This controls what each residue is allowed to "see" during decoding.



Options include:



- `auto_regressive`

- `unconditional`

- `conditional`

- `conditional_minus_self`



This is an expert control. Keep the default unless you know why another pattern is needed.



### `mpnn_initialize_with_ground_truth`



This decides whether sequence embeddings start from the known input sequence or from an unknown/blank state.



For ordinary design runs, the default is usually appropriate.



### `mpnn_atomize_side_chains`



Relevant mainly for `ligand_mpnn` and advanced fixed-residue contexts.



Most users can leave this off.



### Defining What MPNN May Change



These fields define the design scope.



- `mpnn_fixed_residues`: exact residues that must not change.

- `mpnn_designed_residues`: exact residues that are allowed to change.

- `mpnn_fixed_chains`: whole chains that must not change.

- `mpnn_designed_chains`: whole chains that are allowed to change.



Use only one of these approaches at a time. They are different ways to say what is mutable.



### Amino-Acid Preferences And Restrictions



These controls let you push the sequence sampler toward or away from specific residue identities.



- `mpnn_bias`: global amino-acid preferences.

- `mpnn_bias_per_residue`: residue-specific amino-acid preferences.

- `mpnn_omit`: amino acids to forbid globally.

- `mpnn_omit_per_residue`: amino acids to forbid at specific residues.

- `mpnn_pair_bias`: pairwise amino-acid preferences.

- `mpnn_pair_bias_per_residue_pair`: residue-pair-specific preferences.



These are powerful but advanced. Use them only when you already know the sequence chemistry you want to encourage or avoid.



### `mpnn_temperature_per_residue`



This lets you vary sampling temperature across positions.



Useful for expert workflows, but not usually needed for first-pass design.



### MPNN Symmetry Controls



These are separate from RFD3 structural symmetry.



- `mpnn_symmetry_residues`: groups of equivalent residues that should be treated symmetrically.

- `mpnn_symmetry_residues_weights`: weights for those symmetry groups.

- `mpnn_homo_oligomer_chains`: treat matching positions across specified chains as equivalent.



Use these when sequence symmetry matters even after backbone generation.



## Boltz-2 Prediction And Ranking Controls



This stage is optional. It uses Boltz-2 as an external prediction engine after sequence design.



In plain language: RFD3 proposes structures, MPNN proposes sequences, and Boltz provides another layer of structural prediction and ranking.



### `boltz_diffusion_samples`



How many Boltz prediction samples to run for each MPNN structure.



- More samples can improve coverage.

- More samples also increase runtime.



### `boltz_seed`



Reproducibility control for Boltz runs.



### `boltz_affinity`



Whether affinity-related properties should be requested and used in ranking.



If turned on, the notebook also tries to incorporate affinity information into the final ranking table.



### `boltz_ligand_chain`



This tells the notebook which chain should be treated as the ligand or binder chain for affinity and interface metrics.



If this chain is wrong, the ranking metrics for burial and contacts may be misleading.



### `boltz_use_msa_server`



Whether to use an external MSA server when preparing Boltz inputs.



### `generate_yamls_only`



Intended meaning: prepare Boltz YAML files without running prediction.



Current status: this control is exposed in the notebook interface, but the implementation should be treated as incomplete and validated carefully before relying on it.



### How The Ranking Works



The ranking table combines several signals into a single `BoltzGen_Score`.



The current notebook uses:



- prediction confidence

- affinity probability

- predicted affinity value

- interface burial via `delta_sasa`

- polar contact counts

- a simple sequence liability score



This is a **heuristic ranking**, not proof that a design will bind or behave well experimentally.



## Output Locations



The notebook writes outputs under the Colab root.



Current implementation paths:



- `/content/outputs/{design_name}/batch_XX/RFD3_designs/` for RFD3 backbone designs

- `/content/outputs/{design_name}/MPNN_outputs/` for MPNN-designed sequences and CIFs

- `/content/outputs/{design_name}/boltz/` for Boltz YAML files and prediction outputs

- `/content/outputs/{design_name}/ranking/boltz_ranking.xlsx` for the final ranking workbook

- `/content/downloads/{design_name}_results.zip` for the packaged ZIP archive



## Common Mistakes And How To Recover



### The run fails immediately with a validation error



Usually this means one of the following:



- a selection string does not match the input structure

- a dictionary selection is malformed

- `partial_t` was used together with `length`

- the input structure was provided but not actually used by `contig`, `unindex`, `ligand`, or `partial_t`



### The model runs, but the outputs are not changing



Check:



- whether `skip_existing = True` reused earlier outputs

- whether you changed only notebook display settings instead of model settings

- whether the same `seed` and the same design settings were reused



### The design does not appear near the intended binding site



Check:



- `select_hotspots`

- `infer_ori_strategy`

- `ori_token`

- whether your target residues are correctly named and selected



### The symmetry job runs out of memory



Try:



- `diffusion_batch_size = 1`

- `low_memory_mode = True`

- fewer concurrent designs



### MPNN gives unrealistic sequence proposals



Try:



- lowering `mpnn_temperature`

- removing unnecessary advanced bias settings

- verifying that the correct model type was chosen

- checking whether the structure still contains unwanted context components



### The Boltz ranking looks strange



Check:



- whether `boltz_ligand_chain` points to the intended binder or ligand chain

- whether Boltz was properly installed and the runtime restarted after installation

- whether you are interpreting `BoltzGen_Score` as a heuristic rather than a guarantee



## Practical Advice



A good experimental workflow is:



1. Make the smallest possible first run.

2. Confirm the geometry is sensible before sampling many sequences.

3. Increase sequence sampling only after the backbone task is behaving as expected.

4. Use Boltz ranking after you already trust the backbone and sequence-design setup.

5. Keep notes on which settings changed between runs.



If you are unsure which field to change, start with `contig`, `length`, `ligand`, and the simplest selection dictionary needed to express your design idea. Leave the diffusion math controls alone unless you are reproducing a published or previously validated protocol.
